# Watershed Post-Processing for All 3 Models

**Run AFTER** all 3 models have finished training.

For every trained model, the same post-processing pipeline is applied:
```
Image → Trained model (YOLOv8s-seg / Mask R-CNN / Faster R-CNN)
       → Bounding boxes + class labels
       → Crop each predicted box (+ padding for context)
       → Watershed segmentation on crop  → precise pixel boundary
       → Place mask back on full-image canvas
```

Results are saved per-model to `runs/watershed_all_models/` for comparison in `Evaluation.ipynb`.

In [ ]:
import torch
import torchvision
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,  MaskRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import cv2, numpy as np, json, os, time
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pycocotools import mask as coco_mask_util
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

BASE_DIR    = r"C:\Users\User\Desktop\Ipynb"
DATASET_DIR = os.path.join(BASE_DIR, "archive", "dataset_v2")
RUNS_DIR    = os.path.join(BASE_DIR, "runs")

YOLO_WEIGHTS       = os.path.join(RUNS_DIR, "yolov8_seg", "taco_v2", "weights", "best.pt")
MASKRCNN_WEIGHTS   = os.path.join(RUNS_DIR, "mask_rcnn",  "best_mask_rcnn.pth")
FASTERRCNN_WEIGHTS = os.path.join(RUNS_DIR, "faster_rcnn","best_faster_rcnn.pth")

OUTPUT_DIR = os.path.join(RUNS_DIR, "watershed_all_models")
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CLASSES = 10       # 9 waste + background
CONF_THRESH = 0.25
PADDING     = 15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

## Load Test Dataset & Shared Helpers

In [ ]:
# ── Test-set metadata ──────────────────────────────────────────────────────────
with open(os.path.join(DATASET_DIR, "test_annotations.json")) as f:
    test_coco_data = json.load(f)

cat_names = {cat["id"]: cat["name"] for cat in test_coco_data["categories"]}

gt_masks_by_image = defaultdict(list)
for ann in test_coco_data["annotations"]:
    gt_masks_by_image[ann["image_id"]].append(ann)

test_image_dir = os.path.join(DATASET_DIR, "images", "test")
img_file_lookup = {img["file_name"]: img for img in test_coco_data["images"]}
test_files = sorted([
    f for f in os.listdir(test_image_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])
print(f"Test images : {len(test_files)}")
print(f"Categories  : {cat_names}")


# ── Core watershed-on-crop ─────────────────────────────────────────────────────
def watershed_on_crop(crop, padding=10):
    """Watershed on a single-object crop. Returns binary mask same H×W as crop."""
    h, w = crop.shape[:2]
    gray     = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    filtered = cv2.bilateralFilter(crop, 9, 75, 75)
    a_chan   = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)[:, :, 1]

    _, thresh_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh_a = cv2.adaptiveThreshold(a_chan, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 15, 2)
    edges = cv2.Canny(gray, 30, 100)

    combined = cv2.bitwise_or(thresh_otsu, thresh_a)
    combined = cv2.add(combined, cv2.dilate(edges, np.ones((2, 2), np.uint8)))
    _, combined = cv2.threshold(combined, 127, 255, cv2.THRESH_BINARY)

    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(combined, cv2.MORPH_OPEN,  kernel, iterations=1)
    cleaned = cv2.morphologyEx(cleaned,  cv2.MORPH_CLOSE, kernel, iterations=2)

    bw = max(3, min(padding // 2, 8))
    border_mask = np.ones((h, w), dtype=np.uint8) * 255
    border_mask[:bw, :] = border_mask[-bw:, :] = 0
    border_mask[:, :bw] = border_mask[:, -bw:] = 0

    sure_bg = cv2.bitwise_or(cv2.dilate(cleaned, kernel, iterations=3), border_mask)

    dist = cv2.distanceTransform(cleaned, cv2.DIST_L2, 5)
    if dist.max() > 0:
        _, sure_fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
        sure_fg = np.uint8(sure_fg)
    else:
        sure_fg = np.zeros((h, w), dtype=np.uint8)
        cx, cy  = w // 2, h // 2
        r = min(cx, cy, w - cx, h - cy) // 2
        if r > 0:
            cv2.circle(sure_fg, (cx, cy), r, 255, -1)

    unknown  = cv2.subtract(sure_bg, sure_fg)
    _, marks = cv2.connectedComponents(sure_fg)
    marks    = marks + 1
    marks[unknown == 255] = 0
    marks    = cv2.watershed(crop, marks)

    mask  = np.zeros((h, w), dtype=np.uint8)
    valid = set(np.unique(marks)) - {-1, 1}
    if valid:
        best = max(valid, key=lambda lbl: np.sum(marks == lbl))
        mask[marks == best] = 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    else:
        mask = cleaned
    return mask


# ── Apply watershed to a list of boxes ────────────────────────────────────────
def apply_watershed_to_boxes(image, boxes, labels, scores, conf=0.25, padding=15):
    """
    boxes : (N,4) array [x1,y1,x2,y2] in image coords
    labels: (N,)  int array — COCO 1-indexed category_id
    scores: (N,)  float array
    Returns list of dicts: {box, class_id, score, mask}
    """
    h, w = image.shape[:2]
    dets = []
    for box, label, score in zip(boxes, labels, scores):
        if score < conf:
            continue
        x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        x1c = max(0, x1 - padding); y1c = max(0, y1 - padding)
        x2c = min(w, x2 + padding); y2c = min(h, y2 + padding)
        if x2c - x1c < 5 or y2c - y1c < 5:
            continue
        crop_mask = watershed_on_crop(image[y1c:y2c, x1c:x2c], padding=padding)
        full_mask = np.zeros((h, w), dtype=np.uint8)
        full_mask[y1c:y2c, x1c:x2c] = crop_mask
        dets.append({"box": [x1, y1, x2, y2], "class_id": int(label),
                     "score": float(score), "mask": full_mask})
    return dets


# ── GT mask helper ─────────────────────────────────────────────────────────────
def build_gt_mask(anns, img_shape):
    combined = np.zeros(img_shape[:2], dtype=np.uint8)
    for ann in anns:
        for seg in ann.get("segmentation", []):
            pts = np.array(seg, dtype=np.float32).reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(combined, [pts], 255)
    return combined


# ── Unified COCO evaluator ─────────────────────────────────────────────────────
def evaluate_watershed_results(detections_by_image, inf_times):
    """
    detections_by_image: {image_id: (image_shape, fname, [detections])}
    Returns metrics dict with keys matching raw-model result files.
    """
    coco_bbox_preds, coco_segm_preds = [], []
    count_errors, binary_ious = [], []
    per_image = []
    img_lookup = {img["id"]: img for img in test_coco_data["images"]}
    gt_counts  = defaultdict(int)
    for ann in test_coco_data["annotations"]:
        gt_counts[ann["image_id"]] += 1

    for image_id, (img_shape, fname, dets) in detections_by_image.items():
        gt_count = gt_counts.get(image_id, 0)
        err = abs(len(dets) - gt_count)
        count_errors.append(err)

        gt_mask   = build_gt_mask(gt_masks_by_image.get(image_id, []), img_shape)
        pred_comb = np.zeros(img_shape[:2], dtype=np.uint8)
        for det in dets:
            pred_comb = cv2.bitwise_or(pred_comb, det["mask"])
        inter = cv2.bitwise_and(pred_comb, gt_mask)
        union = cv2.bitwise_or(pred_comb, gt_mask)
        binary_ious.append(cv2.countNonZero(inter) / max(cv2.countNonZero(union), 1))

        per_image.append({"file": fname, "gt_count": gt_count,
                          "pred_count": len(dets), "count_error": err})

        img_info = img_lookup.get(image_id, {})
        orig_h   = img_info.get("height", img_shape[0])
        orig_w   = img_info.get("width",  img_shape[1])
        for det in dets:
            x1, y1, x2, y2 = det["box"]
            coco_bbox_preds.append({
                "image_id": image_id, "category_id": det["class_id"],
                "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                "score": det["score"],
            })
            m = det["mask"].copy()
            if m.shape != (orig_h, orig_w):
                m = cv2.resize(m, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            rle = coco_mask_util.encode(np.asfortranarray(m))
            rle["counts"] = rle["counts"].decode("utf-8")
            coco_segm_preds.append({
                "image_id": image_id, "category_id": det["class_id"],
                "segmentation": rle, "score": det["score"],
            })

    gt_coco = COCO()
    gt_coco.dataset = test_coco_data
    gt_coco.createIndex()

    box_map50 = box_map50_95 = mask_map50 = mask_map50_95 = 0.0
    if coco_bbox_preds:
        dt = gt_coco.loadRes(coco_bbox_preds)
        ev = COCOeval(gt_coco, dt, "bbox")
        ev.evaluate(); ev.accumulate(); ev.summarize()
        box_map50, box_map50_95 = float(ev.stats[1]), float(ev.stats[0])
    if coco_segm_preds:
        dt = gt_coco.loadRes(coco_segm_preds)
        ev = COCOeval(gt_coco, dt, "segm")
        ev.evaluate(); ev.accumulate(); ev.summarize()
        mask_map50, mask_map50_95 = float(ev.stats[1]), float(ev.stats[0])

    n = len(count_errors)
    return {
        "metrics": {
            "box_map50":         box_map50,
            "box_map50_95":      box_map50_95,
            "mask_map50":        mask_map50,
            "mask_map50_95":     mask_map50_95,
            "binary_iou":        float(np.mean(binary_ious)),
            "counting_mae":      float(np.mean(count_errors)),
            "counting_within_1": float(sum(1 for e in count_errors if e <= 1) / n * 100),
            "counting_within_3": float(sum(1 for e in count_errors if e <= 3) / n * 100),
            "avg_inference_ms":  float(np.mean(inf_times[1:]) if len(inf_times) > 1 else inf_times[0]),
        },
        "per_image": per_image,
    }


print("Helpers loaded. Test images:", len(test_files))

## Model 1: YOLOv8s-seg + Watershed

YOLOv8 provides bounding boxes; we discard its built-in low-res masks and replace them with watershed masks.

In [ ]:
yolo_model = YOLO(YOLO_WEIGHTS)
print(f"Loaded: {YOLO_WEIGHTS}")

yolo_by_image, yolo_times = {}, []

for fname in tqdm(test_files, desc="YOLO+Watershed"):
    image = cv2.imread(os.path.join(test_image_dir, fname))
    if image is None:
        continue
    img_info = img_file_lookup.get(fname)
    if img_info is None:
        continue
    image_id = img_info["id"]

    t0  = time.perf_counter()
    res = yolo_model(image, imgsz=1024, conf=CONF_THRESH, verbose=False)[0]

    if res.boxes and len(res.boxes) > 0:
        boxes  = res.boxes.xyxy.cpu().numpy()
        labels = res.boxes.cls.cpu().numpy().astype(int) + 1   # 0-idx → COCO 1-idx
        scores = res.boxes.conf.cpu().numpy()
    else:
        boxes = np.zeros((0, 4)); labels = np.zeros(0, int); scores = np.zeros(0)

    dets = apply_watershed_to_boxes(image, boxes, labels, scores, CONF_THRESH, PADDING)
    yolo_times.append((time.perf_counter() - t0) * 1000)
    yolo_by_image[image_id] = (image.shape, fname, dets)

print("\nEvaluating YOLO+Watershed …")
yolo_ws_data = evaluate_watershed_results(yolo_by_image, yolo_times)

print(f"\n{'='*50}")
print("YOLOv8s-seg + Watershed")
print(f"{'='*50}")
for k, v in yolo_ws_data["metrics"].items():
    print(f"  {k:25s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

with open(os.path.join(OUTPUT_DIR, "yolo_watershed_results.json"), "w") as f:
    json.dump(yolo_ws_data, f, indent=2)
print("Saved yolo_watershed_results.json")
yolo_ws_metrics = yolo_ws_data["metrics"]

## Model 2: Mask R-CNN + Watershed

Mask R-CNN already produces masks; watershed refines those boundaries at pixel level.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TACOTestDataset(Dataset):
    """Minimal test dataset for torchvision models."""
    def __init__(self, annotation_file, image_dir, max_size=1024):
        import json as _j
        with open(annotation_file) as f:
            data = _j.load(f)
        self.image_dir  = image_dir
        self.max_size   = max_size
        self.valid_imgs = [img for img in data["images"]
                           if os.path.exists(os.path.join(image_dir, img["file_name"]))]

    def __len__(self): return len(self.valid_imgs)

    def __getitem__(self, idx):
        info  = self.valid_imgs[idx]
        image = cv2.imread(os.path.join(self.image_dir, info["file_name"]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w  = image.shape[:2]
        scale = min(self.max_size / max(h, w), 1.0)
        if scale < 1.0:
            image = cv2.resize(image, (int(w * scale), int(h * scale)))
        img_t = torch.as_tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
        return img_t, info["id"], info["file_name"], h, w

def collate_raw(batch):
    imgs, ids, fnames, hs, ws = zip(*batch)
    return list(imgs), list(ids), list(fnames), list(hs), list(ws)

test_ds     = TACOTestDataset(os.path.join(DATASET_DIR, "test_annotations.json"),
                               os.path.join(DATASET_DIR, "images", "test"))
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                         collate_fn=collate_raw, num_workers=0, pin_memory=True)

# ── Build and load Mask R-CNN ──────────────────────────────────────────────────
mrcnn = maskrcnn_resnet50_fpn_v2(weights=None)
in_f  = mrcnn.roi_heads.box_predictor.cls_score.in_features
mrcnn.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASSES)
in_fm = mrcnn.roi_heads.mask_predictor.conv5_mask.in_channels
mrcnn.roi_heads.mask_predictor = MaskRCNNPredictor(in_fm, 256, NUM_CLASSES)

ckpt = torch.load(MASKRCNN_WEIGHTS, map_location=DEVICE)
mrcnn.load_state_dict(ckpt["model_state_dict"])
mrcnn.to(DEVICE).eval()
print(f"Loaded Mask R-CNN epoch {ckpt['epoch']} val_loss={ckpt['val_loss']:.4f}")

mrcnn_by_image, mrcnn_times = {}, []

with torch.no_grad():
    for imgs, ids, fnames, hs_orig, ws_orig in tqdm(test_loader, desc="MaskRCNN+Watershed"):
        imgs_gpu = [x.to(DEVICE) for x in imgs]
        t0 = time.perf_counter()
        with torch.amp.autocast("cuda"):
            outputs = mrcnn(imgs_gpu)
        inf_ms = (time.perf_counter() - t0) * 1000

        for out, image_id, fname, h_orig, w_orig in zip(outputs, ids, fnames, hs_orig, ws_orig):
            orig = cv2.imread(os.path.join(test_image_dir, fname))
            if orig is None: continue

            h_model, w_model = imgs[0].shape[1], imgs[0].shape[2]
            sx = w_orig / w_model; sy = h_orig / h_model

            boxes  = out["boxes"].cpu().numpy()
            labels = out["labels"].cpu().numpy()
            scores = out["scores"].cpu().numpy()
            boxes_orig = boxes * np.array([sx, sy, sx, sy])

            dets = apply_watershed_to_boxes(orig, boxes_orig, labels, scores, CONF_THRESH, PADDING)
            mrcnn_times.append(inf_ms)
            mrcnn_by_image[image_id] = (orig.shape, fname, dets)

print("\nEvaluating Mask R-CNN + Watershed …")
mrcnn_ws_data = evaluate_watershed_results(mrcnn_by_image, mrcnn_times)

print(f"\n{'='*50}")
print("Mask R-CNN + Watershed")
print(f"{'='*50}")
for k, v in mrcnn_ws_data["metrics"].items():
    print(f"  {k:25s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

with open(os.path.join(OUTPUT_DIR, "maskrcnn_watershed_results.json"), "w") as f:
    json.dump(mrcnn_ws_data, f, indent=2)
print("Saved maskrcnn_watershed_results.json")
mrcnn_ws_metrics = mrcnn_ws_data["metrics"]

## Model 3: Faster R-CNN + Watershed

Faster R-CNN produces boxes only (no masks). Watershed gives it segmentation capability.

In [ ]:
# ── Build and load Faster R-CNN ───────────────────────────────────────────────
frcnn = fasterrcnn_resnet50_fpn_v2(weights=None)
in_f  = frcnn.roi_heads.box_predictor.cls_score.in_features
frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASSES)

ckpt = torch.load(FASTERRCNN_WEIGHTS, map_location=DEVICE)
frcnn.load_state_dict(ckpt["model_state_dict"])
frcnn.to(DEVICE).eval()
print(f"Loaded Faster R-CNN epoch {ckpt['epoch']} val_loss={ckpt['val_loss']:.4f}")

frcnn_by_image, frcnn_times = {}, []

with torch.no_grad():
    for imgs, ids, fnames, hs_orig, ws_orig in tqdm(test_loader, desc="FasterRCNN+Watershed"):
        imgs_gpu = [x.to(DEVICE) for x in imgs]
        t0 = time.perf_counter()
        with torch.amp.autocast("cuda"):
            outputs = frcnn(imgs_gpu)
        inf_ms = (time.perf_counter() - t0) * 1000

        for out, image_id, fname, h_orig, w_orig in zip(outputs, ids, fnames, hs_orig, ws_orig):
            orig = cv2.imread(os.path.join(test_image_dir, fname))
            if orig is None: continue

            h_model, w_model = imgs[0].shape[1], imgs[0].shape[2]
            sx = w_orig / w_model; sy = h_orig / h_model

            boxes  = out["boxes"].cpu().numpy()
            labels = out["labels"].cpu().numpy()
            scores = out["scores"].cpu().numpy()
            boxes_orig = boxes * np.array([sx, sy, sx, sy])

            dets = apply_watershed_to_boxes(orig, boxes_orig, labels, scores, CONF_THRESH, PADDING)
            frcnn_times.append(inf_ms)
            frcnn_by_image[image_id] = (orig.shape, fname, dets)

print("\nEvaluating Faster R-CNN + Watershed …")
frcnn_ws_data = evaluate_watershed_results(frcnn_by_image, frcnn_times)

print(f"\n{'='*50}")
print("Faster R-CNN + Watershed")
print(f"{'='*50}")
for k, v in frcnn_ws_data["metrics"].items():
    print(f"  {k:25s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

with open(os.path.join(OUTPUT_DIR, "fasterrcnn_watershed_results.json"), "w") as f:
    json.dump(frcnn_ws_data, f, indent=2)
print("Saved fasterrcnn_watershed_results.json")
frcnn_ws_metrics = frcnn_ws_data["metrics"]

## Watershed Summary & Visualization

In [ ]:
import pandas as pd

all_ws = [
    ("YOLOv8s-seg+WS", yolo_ws_metrics),
    ("Mask R-CNN+WS",  mrcnn_ws_metrics),
    ("Faster R-CNN+WS",frcnn_ws_metrics),
]

rows = []
for name, m in all_ws:
    rows.append({
        "Model":          name,
        "Box mAP@50":     f"{m['box_map50']:.3f}",
        "Mask mAP@50":    f"{m['mask_map50']:.3f}",
        "Binary IoU":     f"{m['binary_iou']:.3f}",
        "Count MAE":      f"{m['counting_mae']:.2f}",
        "Within ±1 (%)":  f"{m['counting_within_1']:.1f}",
        "Inf (ms)":       f"{m['avg_inference_ms']:.0f}",
    })

df = pd.DataFrame(rows).set_index("Model")
print("\n" + "="*70)
print("WATERSHED POST-PROCESSING SUMMARY (all 3 models on test set)")
print("="*70)
print(df.to_string())
print("="*70)

# Bar charts
models   = [r["Model"] for r in rows]
mask_map = [float(r["Mask mAP@50"]) for r in rows]
bio_iou  = [float(r["Binary IoU"])  for r in rows]
cnt_mae  = [float(r["Count MAE"])   for r in rows]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#4C72B0", "#55A868", "#C44E52"]

axes[0].bar(models, mask_map, color=colors)
axes[0].set_title("Mask mAP@50 (watershed)"); axes[0].set_ylim(0, 1)
axes[1].bar(models, bio_iou, color=colors)
axes[1].set_title("Binary Mask IoU"); axes[1].set_ylim(0, 1)
axes[2].bar(models, cnt_mae, color=colors)
axes[2].set_title("Counting MAE (lower=better)")

for ax in axes:
    ax.tick_params(axis="x", rotation=10); ax.grid(axis="y", alpha=0.4)

plt.suptitle("Watershed Post-Processing: All 3 Models", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "watershed_comparison.png"), dpi=150)
plt.show()
print("Saved watershed_comparison.png")

# Qualitative visualization
CMAP = plt.cm.tab10.colors
sample_fname = test_files[0]
sample_image = cv2.imread(os.path.join(test_image_dir, sample_fname))
sample_rgb   = cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB)
sample_id    = img_file_lookup[sample_fname]["id"]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
axes[0].imshow(sample_rgb); axes[0].set_title("Original"); axes[0].axis("off")

for ax, (label, by_img) in zip(axes[1:], [
        ("YOLOv8s-seg+WS", yolo_by_image),
        ("Mask R-CNN+WS",  mrcnn_by_image),
        ("Faster R-CNN+WS",frcnn_by_image),
]):
    overlay = sample_rgb.copy()
    entry   = by_img.get(sample_id)
    dets    = entry[2] if entry else []
    for i, det in enumerate(dets):
        c = [int(x * 255) for x in CMAP[i % len(CMAP)][:3]]
        colored = np.zeros_like(overlay)
        colored[det["mask"] == 255] = c
        overlay = cv2.addWeighted(overlay, 1.0, colored, 0.35, 0)
        cnts, _ = cv2.findContours(det["mask"], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, cnts, -1, c, 2)
    ax.imshow(overlay)
    ax.set_title(f"{label}\n({len(dets)} objects)")
    ax.axis("off")

plt.suptitle(f"Sample: {sample_fname}", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "watershed_qualitative.png"), dpi=120)
plt.show()
print("Saved watershed_qualitative.png")